In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import functools
import joblib
import os
import pathlib

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.multitest

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [ ]:
BASE = common_data.DATA / '05_pseudobulk/50a_other_degs'

In [5]:
adata = sc.read_h5ad(common_data.SC_NORM)

In [6]:
categorical = common_data.get_sc_categorical_covariates()
numerical = common_data.get_sc_numerical_covariates()
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [7]:
numerical['cohort'] = sc_labels.set_index('bal_barcode').cohort

In [8]:
categorical['cohort'] = sc_labels.set_index('bal_barcode').cohort

In [9]:
numerical['age_over_64'] = numerical.Age.gt(64) & numerical.cohort.eq('SCRIPT')
numerical['steroid_over_median'] = numerical.cumulative_icu_steroid_dose_until_today.gt(
    numerical.cumulative_icu_steroid_dose_until_today.median()
) & numerical.cohort.eq('SCRIPT')
numerical['vent_q1'] = (
    numerical.days_on_ventilator.lt(numerical.days_on_ventilator.quantile(0.25))
    & numerical.cohort.eq('SCRIPT')
)
numerical['vent_q4'] = (
    numerical.days_on_ventilator.gt(numerical.days_on_ventilator.quantile(0.75))
    & numerical.cohort.eq('SCRIPT')
)

In [10]:
tasks = [
    'gram+_vs_gram-',
    'pseudomonas_vs_gram-',
    'age',
    'sex',
    'immunocompromised',
    'days_on_vent',
    'steroid_dose'
]

In [11]:
def get_positive_label(task, cat, num):
    if task == 'gram+_vs_gram-':
        return cat.Pathogen_groups.isin(['Gram-*', 'Pseudomonas aeruginosa'])
    if task == 'pseudomonas_vs_gram-':
        return cat.Pathogen_groups.eq('Gram-*')
    if task == 'age':
        return num.age_over_64 & num.cohort.eq('SCRIPT')
    if task == 'sex':
        return cat.Sex.eq('Female') & num.cohort.eq('SCRIPT')
    if task == 'immunocompromised':
        return cat.Immunocompromised_flag.eq('1')
    if task == 'days_on_vent':
        return num.vent_q4
    if task == 'steroid_dose':
        return num.steroid_over_median & num.cohort.eq('SCRIPT')
    raise ValueError(f'Unknown task {task}')

In [12]:
def get_negative_label(task, cat, num):
    if task == 'gram+_vs_gram-':
        return cat.Pathogen_groups.eq('Gram+')
    if task == 'pseudomonas_vs_gram-':
        return cat.Pathogen_groups.eq('Pseudomonas aeruginosa')
    if task == 'age':
        return ~num.age_over_64 & num.cohort.eq('SCRIPT')
    if task == 'sex':
        return cat.Sex.eq('Male') & num.cohort.eq('SCRIPT')
    if task == 'immunocompromised':
        return cat.Immunocompromised_flag.eq('0')
    if task == 'days_on_vent':
        return num.vent_q1
    if task == 'steroid_dose':
        return ~num.steroid_over_median & num.cohort.eq('SCRIPT')
    raise ValueError(f'Unknown task {task}')

In [13]:
%%time
PSEUDOBULK_CELLS = 50
PSEUDOBULK_EXPR_IN_GROUP = 0.8
genes_to_keep_per_task = {}
for task in tasks:
    task_genes = {}
    for ct in adata.obs.Level_6.unique():
        genes = None
        pos_idx = get_positive_label(task, categorical, numerical)
        neg_idx = get_negative_label(task, categorical, numerical)
        pos_samples = categorical.index[pos_idx]
        neg_samples = categorical.index[neg_idx]
        for samples in [pos_samples, neg_samples]:
            pseudobulks = []
            for sample in samples:
                idx = adata.obs.Level_6.eq(ct) & adata.obs.bal_barcode.eq(sample)
                if idx.sum() < PSEUDOBULK_CELLS:
                    continue
                pseudobulks.append(adata.raw.X[idx, :].sum(axis=0).A1)
            if len(pseudobulks) == 0:
                continue
            pseudobulks = pd.DataFrame(pseudobulks, columns=adata.raw.var_names)
            expr_frac = (pseudobulks > 0).sum(axis=0) / pseudobulks.shape[0]
            group_genes = pseudobulks.columns[expr_frac.ge(PSEUDOBULK_EXPR_IN_GROUP)]
            if genes is None:
                genes = group_genes.to_numpy()
            else:
                genes = np.union1d(genes, group_genes)
        task_genes[ct] = genes
    genes_to_keep_per_task[task] = task_genes

CPU times: user 3min 29s, sys: 8.14 s, total: 3min 37s
Wall time: 3min 37s


In [14]:
{k: len(v) for k, v in genes_to_keep_per_task['age'].items() if v is not None}

{'CD4 T cells': 10005,
 'CD8 T cells': 10033,
 'Mast cells': 8867,
 'NUPR1+ Macs': 11991,
 'B cells': 8742,
 'MRC1+C1QA+': 11507,
 'DC2': 10269,
 'MRC1+C1QA-': 11704,
 'Proliferating CD4 T cells': 10392,
 'Proliferating CD8 T cells': 10058,
 'Classical monocytes-2 IL1B': 9846,
 'Secretory cells': 12276,
 'Proliferating NUPR1+ Macs': 11294,
 'Tregs': 8769,
 'DC1': 10197,
 'Ionocytes': 13684,
 'Ciliated cells': 11931,
 'gdT cells': 9011,
 'Migratory DC': 11489,
 'Interstitial macrophages': 9238,
 'Hematopoietic stem cells': 16487,
 'Classical monocytes-1 CCR2': 8964,
 'pDC': 9879,
 'AT1 and AT2': 12371,
 'Proliferating plasma cells': 10581,
 'Non-classical monocytes': 9415,
 'Plasma cells': 9680,
 'Proliferating gdT cells': 9841}

In [15]:
for task in tasks:
    genes_to_keep_per_task[task]['Perivascular macrophages'] = genes_to_keep_per_task[task]['Interstitial macrophages']

In [ ]:
PADJ_CUTOFF = 0.05
class ComparisonInfo:
    def __init__(self, control, condition, genes, genes_to_keep, cell_type_info):
        self.control = control
        self.condition = condition
        self.genes_raw = genes
        self.cell_type_info = cell_type_info
        self.filter_genes(genes_to_keep)

    def filter_genes(self, genes_to_keep):
        filtered_degs = self.genes_raw.loc[self.genes_raw.index.isin(genes_to_keep), :].copy()
        # use padj, see https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html#indfilttheory
        # Except in case of MRC1+C1QA+ AM in age comparison where DESeq2 estimation fails
        if (
                self.cell_type_info.task_info.pathname == 'age'
                and self.cell_type_info.name == 'MRC1+C1QA+'
            ):
            base_mean_threshold = self.genes_raw.baseMean.quantile(0.3)
            filtered_degs = filtered_degs.loc[filtered_degs.baseMean.gt(base_mean_threshold)].copy()
        else:
            filtered_degs = filtered_degs.loc[filtered_degs.padj.notna()].copy()
        # recompute FDR correction on the filtered genes:
        filtered_degs['padj'] = statsmodels.stats.multitest.fdrcorrection(
            filtered_degs.pvalue,
            alpha=PADJ_CUTOFF
        )[1]
        # recompute gene status based on new `padj`
        filtered_degs['sign'] = ''
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.gt(0),
            'sign'
        ] = f'Up in {self.condition}'
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.lt(0),
            'sign'
        ] = f'Up in {self.control}'
        self.genes = filtered_degs


class CellTypeInfo:
    def __init__(self, path, task_info, genes_to_keep):
        self.path = path
        self.task_info = task_info
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.name = self.meta.cell_type.values[0]

        self.load_comparisons(genes_to_keep)

    def load_comparisons(self, genes_to_keep):
        for run in self.path.glob('**/degs.csv'):
            self.comparisons.append(
                ComparisonInfo(
                    self.task_info.column_values[0],
                    self.task_info.column_values[1],
                    pd.read_csv(run, index_col=0),
                    genes_to_keep[self.name],
                    self
                )
            )

    @property
    def n_comparisons(self):
        return len(self.comparisons)

In [24]:
class TaskData:
    def __init__(self, task, task_info):
        self.task = task
        self.task_info = task_info
        self.info = {}

In [25]:
task_pairs = {
    'gram+_vs_gram-': ('Gram+', 'Gram–'),
    'pseudomonas_vs_gram-': ('Pseudomonas', 'Other Gram–'),
    'age': ('Age < 65', 'Age >= 65'),
    'sex': ('Male', 'Female'),
    'immunocompromised': ('Not immunocompromised', 'Immunocompromised'),
    'days_on_vent': ('< 2 vent days', '> 16 vent days'),
    'steroid_dose': ('<= 600 steroid dose', '> 600 steroid dose')
}

In [31]:
%%time
data = {}
for task in tasks:
    task_info = common_data.TaskInfo(
        pathname=task,
        column=None,
        column_values=task_pairs[task],
        split_column=None
    )
    task_data = TaskData(task, task_info)
    for cell_type_path in sorted((BASE / task_info.pathname).iterdir()):
        if cell_type_path.name.startswith('.') or cell_type_path.name.startswith('_'):
            continue
        if not cell_type_path.is_dir():
            continue
        if not (cell_type_path / 'meta.csv').exists():
            continue
        info = CellTypeInfo(cell_type_path, task_info, genes_to_keep_per_task[task])
        if info.n_comparisons > 0:
            task_data.info[cell_type_path.name] = info
    if len(task_data.info) > 0:
        data[task] = task_data

CPU times: user 2.42 s, sys: 146 ms, total: 2.57 s
Wall time: 9.72 s


In [32]:
data.keys()

dict_keys(['gram+_vs_gram-', 'pseudomonas_vs_gram-', 'age', 'sex', 'immunocompromised', 'days_on_vent', 'steroid_dose'])

In [ ]:
joblib.dump(data, '50c_deg_data.joblib')

Save filtered DEGs as csv to run GSEA on them

In [ ]:
BASE = BASE.parent / '50c_degs'
for _, task in data.items():
    for k, ct_info in task.info.items():
        comp = ct_info.comparisons[0]
        deg_path = BASE / task.task_info.pathname / k / 'degs.csv'
        if deg_path.exists():
            print(f'File {deg_path} already exists, skipping')
            continue
        # Threshold GSEA analysis to at least 1000 genes in comparison
        if comp.genes.shape[0] < 1000:
            print(f'Skipping {ct_info.name} for {task.task_info.pathname}')
            continue
        ct_path = BASE / task.task_info.pathname / k
        os.makedirs(ct_path, exist_ok=True)
        comp.genes.sort_values('log2FoldChange').to_csv(deg_path)